# RetailMind AI - Day 9: RNN / LSTM Demand Forecasting

This increment starts Phase 3 by forecasting next-period product/store demand with an LSTM.

**Dataset expected:** Kaggle Store Item Demand (`train.csv`) or Rossmann Store Sales (`train.csv`). Put the CSV under `RetailMind/data/demand/`. The notebook detects either schema automatically.

**Today's libraries:** pandas, TensorFlow / Keras. The notebook uses TensorFlow operations for scaling and MAE/RMSE so no extra ML library is required.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import json
import pandas as pd
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)
current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == 'notebooks' else current
DATA_DIR = PROJECT_ROOT / 'data' / 'demand'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR = MODELS_DIR / 'day9_forecast_outputs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Demand-data folder:', DATA_DIR)
print('TensorFlow:', tf.__version__)

I0000 00:00:1794759807.060602 2699391 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794759808.806222 2699391 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794759813.484476 2699391 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Project root: /home/jovyan/work/RetailMind
Demand-data folder: /home/jovyan/work/RetailMind/data/demand
TensorFlow: 2.21.0


## 2. Locate and load the sales CSV

Place the Kaggle training CSV in `data/demand/`. If several CSV files are present, set `CSV_PATH` manually to the required file.

In [2]:
csv_files = sorted(DATA_DIR.glob('*.csv')) if DATA_DIR.exists() else []
if not csv_files:
    raise FileNotFoundError(
        'No CSV found. Create RetailMind/data/demand/ and place the Kaggle train.csv there.'
    )
CSV_PATH = csv_files[0]
df = pd.read_csv(CSV_PATH)
print('Loaded:', CSV_PATH)
print('Shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())

Loaded: /home/jovyan/work/RetailMind/data/demand/store_item_demand.csv
Shape: (282049, 4)


,date,store,item,sales
0,2013-01-01,1.0,1.0,13.0
1,2013-01-02,1.0,1.0,11.0
2,2013-01-03,1.0,1.0,14.0
3,2013-01-04,1.0,1.0,13.0
4,2013-01-05,1.0,1.0,10.0


Columns: ['date', 'store', 'item', 'sales']


## 3. Detect dataset schema and create one clean store/product time series

- Store Item Demand schema: `date`, `store`, `item`, `sales`
- Rossmann schema: `Date`, `Store`, `Sales`

To keep the Day 9 LSTM practical on the work laptop, the notebook forecasts a single store/item series. This demonstrates the demand module cleanly and can later be generalized across stores/items.

In [3]:
columns_lower = {c.lower(): c for c in df.columns}

if {'date', 'store', 'item', 'sales'}.issubset(columns_lower):
    DATASET_NAME = 'Store Item Demand'
    date_col = columns_lower['date']
    store_col = columns_lower['store']
    item_col = columns_lower['item']
    sales_col = columns_lower['sales']
    selected_store = sorted(df[store_col].dropna().unique())[0]
    selected_item = sorted(df[item_col].dropna().unique())[0]
    series_df = df[(df[store_col] == selected_store) & (df[item_col] == selected_item)][[date_col, sales_col]].copy()
    SERIES_ID = f'store_{selected_store}_item_{selected_item}'
elif {'date', 'store', 'sales'}.issubset(columns_lower):
    DATASET_NAME = 'Rossmann Store Sales'
    date_col = columns_lower['date']
    store_col = columns_lower['store']
    sales_col = columns_lower['sales']
    selected_store = sorted(df[store_col].dropna().unique())[0]
    series_df = df[df[store_col] == selected_store][[date_col, sales_col]].copy()
    SERIES_ID = f'store_{selected_store}'
else:
    raise ValueError(
        'Unsupported schema. Expected Store Item Demand (date/store/item/sales) or Rossmann (Date/Store/Sales).'
    )

series_df[date_col] = pd.to_datetime(series_df[date_col])
series_df[sales_col] = pd.to_numeric(series_df[sales_col], errors='coerce')
series_df = series_df.dropna().sort_values(date_col)
print('Dataset:', DATASET_NAME)
print('Series:', SERIES_ID)
print('Rows:', len(series_df))
display(series_df.head())

Dataset: Store Item Demand
Series: store_1.0_item_1.0
Rows: 1826


,date,sales
0,2013-01-01,13.0
1,2013-01-02,11.0
2,2013-01-03,14.0
3,2013-01-04,13.0
4,2013-01-05,10.0


## 4. Resample to a clean daily demand series with pandas

The selected store/product is aggregated to daily sales. Missing calendar dates are filled with zero demand.

In [4]:
daily = (
    series_df.set_index(date_col)[sales_col]
    .resample('D')
    .sum()
    .asfreq('D', fill_value=0)
    .astype('float32')
)
print('Date range:', daily.index.min().date(), 'to', daily.index.max().date())
print('Daily observations:', len(daily))
display(daily.head())
daily.to_csv(OUTPUT_DIR / 'daily_demand_series.csv', header=['sales'])

Date range: 2013-01-01 to 2017-12-31
Daily observations: 1826


date
2013-01-01    13.0
2013-01-02    11.0
2013-01-03    14.0
2013-01-04    13.0
2013-01-05    10.0
Freq: D, Name: sales, dtype: float32

## 5. Hold out the most recent demand for testing

This is a time-series split, not a random split. The most recent 20% of observations are reserved for testing so the model is evaluated on future periods.

In [5]:
values = tf.convert_to_tensor(daily.values, dtype=tf.float32)
split_index = int(len(daily) * 0.80)
train_values = values[:split_index]
test_values = values[split_index:]
train_dates = daily.index[:split_index]
test_dates = daily.index[split_index:]
print('Training observations:', len(train_values))
print('Test observations:', len(test_values))
print('Test starts:', test_dates[0].date())

Training observations: 1460
Test observations: 366
Test starts: 2016-12-31


E0000 00:00:1794759817.009215 2699391 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 6. Scale using training statistics only

In [6]:
train_mean = tf.reduce_mean(train_values)
train_std = tf.math.reduce_std(train_values)
train_std = tf.where(train_std == 0, tf.constant(1.0), train_std)
scaled_values = (values - train_mean) / train_std
print('Training mean:', float(train_mean))
print('Training std:', float(train_std))

Training mean: 19.422603607177734
Training std: 6.572579860687256


## 7. Frame supervised windows: look-back → next-step demand

Each training sample uses the previous 30 daily observations to predict the next day's demand.

In [7]:
LOOKBACK = 30
def make_windows(series, lookback):
    windows = tf.signal.frame(series, frame_length=lookback + 1, frame_step=1)
    X = windows[:, :-1]
    y = windows[:, -1]
    return X[..., tf.newaxis], y

X_all, y_all = make_windows(scaled_values, LOOKBACK)
target_indices = tf.range(LOOKBACK, len(values))
train_mask = target_indices < split_index
test_mask = target_indices >= split_index
X_train = tf.boolean_mask(X_all, train_mask)
y_train = tf.boolean_mask(y_all, train_mask)
X_test = tf.boolean_mask(X_all, test_mask)
y_test = tf.boolean_mask(y_all, test_mask)
test_target_dates = daily.index[LOOKBACK:][test_mask.numpy()]
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_test:', X_test.shape, 'y_test:', y_test.shape)

X_train: (1430, 30, 1) y_train: (1430,)
X_test: (366, 30, 1) y_test: (366,)


## 8. Build the LSTM demand forecaster

In [8]:
lstm_model = keras.Sequential([
    keras.layers.Input(shape=(LOOKBACK, 1)),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.20),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,897 (19.13 KB)

 Trainable params: 4,897 (19.13 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train with chronological validation and EarlyStopping

In [9]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
history = lstm_model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.20,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)
print('Epochs run:', len(history.history['loss']))
print('Best validation loss:', round(min(history.history['val_loss']), 4))

Epoch 1/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.8114 - mae: 0.7098 - val_loss: 1.0471 - val_mae: 0.8053
Epoch 2/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.6726 - mae: 0.6425 - val_loss: 0.8248 - val_mae: 0.7217
Epoch 3/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6429 - mae: 0.6270 - val_loss: 0.7452 - val_mae: 0.6898
Epoch 4/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6310 - mae: 0.6197 - val_loss: 0.7320 - val_mae: 0.6852
Epoch 5/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6471 - mae: 0.6280 - val_loss: 0.7410 - val_mae: 0.6874
Epoch 6/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6341 - mae: 0.6177 - val_loss: 0.7261 - val_mae: 0.6820
Epoch 7/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6325 - mae: 0.6204 - val_loss: 0.7272 - val_mae: 0.6816
Epoch 8/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6305 - mae: 0.6185 - val_loss: 0.7241 - val_mae: 0.6807
Epoch 9/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6366 - mae

## 10. Predict and reverse the scaling

In [11]:
pred_scaled = tf.squeeze(lstm_model.predict(X_test, verbose=0), axis=1)
predicted_sales = pred_scaled * train_std + train_mean
actual_sales = y_test * train_std + train_mean
predicted_sales = tf.maximum(predicted_sales, 0.0)
forecast_df = pd.DataFrame({
    'date': test_target_dates,
    'actual_sales': actual_sales.numpy(),
    'predicted_sales': predicted_sales.numpy()
})
display(forecast_df.head(10))

,date,actual_sales,predicted_sales
0,2016-12-31,14.0,17.868423
1,2017-01-01,19.0,17.351528
2,2017-01-02,15.0,17.638821
3,2017-01-03,10.0,17.344261
4,2017-01-04,16.0,16.476023
5,2017-01-05,14.0,16.596380
6,2017-01-06,24.0,16.358042
7,2017-01-07,14.0,17.377075
8,2017-01-08,20.0,16.908316
9,2017-01-09,18.0,17.359232


## 11. Evaluate with MAE and RMSE

In [12]:
errors = predicted_sales - actual_sales
mae = tf.reduce_mean(tf.abs(errors))
rmse = tf.sqrt(tf.reduce_mean(tf.square(errors)))
print('MAE:', round(float(mae), 4))
print('RMSE:', round(float(rmse), 4))

MAE: 4.5853
RMSE: 5.8225


## 12. Predicted vs actual sanity-check

The table below shows the most recent predictions. It is the notebook-friendly alternative to adding an extra plotting library beyond today's stated libraries.

In [13]:
comparison = forecast_df.tail(20).copy()
comparison['error'] = comparison['predicted_sales'] - comparison['actual_sales']
display(comparison.round(2))

,date,actual_sales,predicted_sales,error
346,2017-12-12,13.0,19.799999,6.80
347,2017-12-13,17.0,18.690001,1.69
348,2017-12-14,14.0,18.389999,4.39
349,2017-12-15,16.0,17.510000,1.51
350,2017-12-16,15.0,17.190001,2.19
351,2017-12-17,22.0,16.830000,-5.17
352,2017-12-18,19.0,17.660000,-1.34
353,2017-12-19,7.0,17.860001,10.86
354,2017-12-20,16.0,16.309999,0.31
355,2017-12-21,12.0,16.430000,4.43


## 13. Save forecast outputs and LSTM model

In [14]:
MODEL_PATH = MODELS_DIR / 'demand_lstm.keras'
FORECAST_PATH = OUTPUT_DIR / 'demand_forecast.csv'
METRICS_PATH = MODELS_DIR / 'day9_demand_metrics.json'
lstm_model.save(MODEL_PATH)
forecast_df.to_csv(FORECAST_PATH, index=False)
metrics = {
    'dataset': DATASET_NAME,
    'series': SERIES_ID,
    'lookback_days': LOOKBACK,
    'train_observations': int(len(train_values)),
    'test_observations': int(len(test_values)),
    'epochs_run': len(history.history['loss']),
    'best_validation_loss': float(min(history.history['val_loss'])),
    'mae': float(mae),
    'rmse': float(rmse),
    'model': 'models/demand_lstm.keras',
    'forecast_output': 'models/day9_forecast_outputs/demand_forecast.csv'
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved model:', MODEL_PATH)
print('Saved forecast:', FORECAST_PATH)
print('Saved metrics:', METRICS_PATH)

Saved model: /home/jovyan/work/RetailMind/models/demand_lstm.keras
Saved forecast: /home/jovyan/work/RetailMind/models/day9_forecast_outputs/demand_forecast.csv
Saved metrics: /home/jovyan/work/RetailMind/models/day9_demand_metrics.json


## 14. Simple demand-module interface

In [15]:
def forecast_next_period(recent_sales):
    if len(recent_sales) != LOOKBACK:
        raise ValueError(f'Provide exactly {LOOKBACK} recent daily sales values.')
    recent = tf.convert_to_tensor(recent_sales, dtype=tf.float32)
    recent_scaled = (recent - train_mean) / train_std
    model_x = tf.reshape(recent_scaled, (1, LOOKBACK, 1))
    next_scaled = tf.squeeze(lstm_model.predict(model_x, verbose=0))
    next_sales = next_scaled * train_std + train_mean
    return max(0.0, float(next_sales))

example_recent_sales = daily.iloc[-LOOKBACK:].tolist()
next_demand = forecast_next_period(example_recent_sales)
print('Next-period demand forecast:', round(next_demand, 2))

Next-period demand forecast: 17.57


## 15. Verify Day 9 artifacts

In [16]:
required = [MODEL_PATH, FORECAST_PATH, METRICS_PATH, OUTPUT_DIR / 'daily_demand_series.csv']
for path in required:
    print(('OK' if path.exists() else 'MISSING'), '-', path.relative_to(PROJECT_ROOT))
assert all(path.exists() for path in required), 'A Day 9 artifact is missing.'
print('Day 9 completed successfully.')

OK - models/demand_lstm.keras
OK - models/day9_forecast_outputs/demand_forecast.csv
OK - models/day9_demand_metrics.json
OK - models/day9_forecast_outputs/daily_demand_series.csv
Day 9 completed successfully.


## 16. README-ready summary

In [17]:
print('DAY 9 - LSTM DEMAND FORECASTING')
print('-' * 60)
print('Dataset:', DATASET_NAME)
print('Series:', SERIES_ID)
print('Look-back:', LOOKBACK, 'days')
print('Training observations:', len(train_values))
print('Test observations:', len(test_values))
print('Epochs run:', len(history.history['loss']))
print('MAE:', f'{float(mae):.4f}')
print('RMSE:', f'{float(rmse):.4f}')
print('Next-period forecast:', f'{next_demand:.2f}')
print('Model: models/demand_lstm.keras')
print('Forecast file: models/day9_forecast_outputs/demand_forecast.csv')

DAY 9 - LSTM DEMAND FORECASTING
------------------------------------------------------------
Dataset: Store Item Demand
Series: store_1.0_item_1.0
Look-back: 30 days
Training observations: 1460
Test observations: 366
Epochs run: 20
MAE: 4.5853
RMSE: 5.8225
Next-period forecast: 17.57
Model: models/demand_lstm.keras
Forecast file: models/day9_forecast_outputs/demand_forecast.csv


## Requirement coverage

- Loaded and resampled sales into a clean per-store/product daily time series with pandas
- Framed supervised look-back → next-step windows
- Scaled the series using training-only statistics
- Built and trained an LSTM next-period forecaster
- Held out the most recent observations for testing
- Evaluated with MAE and RMSE
- Produced a predicted-vs-actual sanity-check table
- Saved the forecaster and forecast output as the demand module feeding future inventory planning

### Business value
The Day 9 module adds forward-looking demand signals to RetailMind. Better demand estimates can support reorder timing, reduce stockouts, and reduce unnecessary overstock.

### Open blocker
This first increment models one store/product series and performs one-step-ahead forecasting. The next iteration can extend the same workflow to multiple series and multi-horizon Seq2Seq + attention forecasting.